In [13]:
import pandas as pd

df = pd.read_csv("../data/M_PL_001.csv", index_col=0)

print(df.shape)
print(df.head())

(84, 101)
                       Alloscirtetica gayi  Alloscirtetica rufitarsis  \
Acaena pinn                              0                          0   
Adesmia aconcaguensis                    0                          0   
Adesmia brachy                           0                          0   
Adesmia conferta                         0                          1   
Adesmia exilis                           0                          0   

                       Alloscirtetica sp1 M_PL_001  Andinus venustus  \
Acaena pinn                                      0                 0   
Adesmia aconcaguensis                            0                 0   
Adesmia brachy                                   0                 0   
Adesmia conferta                                 1                 0   
Adesmia exilis                                   0                 0   

                       Anthidium adriani  Anthidium chilensis  \
Acaena pinn                            0             

In [14]:
print(df.isnull().sum().sum())

0


In [15]:
df = df.fillna(0)

In [16]:
df_binary = (df > 0).astype(int)

In [17]:
import networkx as nx

G = nx.Graph()

for plant in df_binary.index:
    for pollinator in df_binary.columns:
        if df_binary.loc[plant, pollinator] == 1:
            G.add_edge(plant, pollinator)

In [18]:
for node in G.nodes():
    if node in df_binary.index:
        G.nodes[node]['type'] = 'plant'
    else:
        G.nodes[node]['type'] = 'pollinator'

In [19]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 185
Edges: 361


In [20]:
from networkx.algorithms import bipartite

print(bipartite.is_bipartite(G))

True


In [23]:
from pathlib import Path

out_path = Path("../output/network_001.gexf")
out_path.parent.mkdir(parents=True, exist_ok=True)

nx.write_gexf(G, out_path)
print(f"Saved: {out_path.resolve()}")

Saved: C:\Users\akash\Videos\Mtech_IIT\2nd sem\network science\project\ns\output\network_001.gexf


In [27]:
import os
import pandas as pd
import networkx as nx

data_folder = "../data/"
graphs = []

for file in sorted(os.listdir(data_folder)):
    
    # ✅ filter only pollination datasets
    if file.startswith("M_PL") and file.endswith(".csv"):
        
        df = pd.read_csv(os.path.join(data_folder, file), index_col=0)
        
        df_numeric = df.apply(pd.to_numeric, errors="coerce").fillna(0)
        df_binary = (df_numeric > 0).astype(int)

        G = nx.Graph()

        for plant in df_binary.index:
            for pollinator in df_binary.columns:
                if df_binary.loc[plant, pollinator] == 1:
                    G.add_edge(plant, pollinator)

        graphs.append(G)

print("Total pollination graphs:", len(graphs))

Total pollination graphs: 174
